In [1]:

import numpy as np
import pandas as pd
import requests
from lib.odds import Odds

%load_ext magic_duckdb

In [2]:
df = Odds().get_sports()


In [6]:
sport = 'basketball_ncaab_championship_winner'
odds_response = requests.get(
    f'https://api.the-odds-api.com/v4/sports/{sport}/odds',
    params={
        'api_key': '2b1fbcaa5cf9bbfca66c986129524664',
        'regions': 'us',
        'markets': 'outrights',
        'oddsFormat': 'decimal',
        'dateFormat': 'iso',
    }
)

if odds_response.status_code != 200:
    print(f'Failed to get odds: status_code {odds_response.status_code}, response body {odds_response.text}')

else:
    odds_json = odds_response.json()
    print('Number of events:', len(odds_json))
    print(odds_json)

    # Check the usage quota
    print('Remaining requests', odds_response.headers['x-requests-remaining'])
    print('Used requests', odds_response.headers['x-requests-used'])

Number of events: 1
[{'id': 'f1598ca8abdce1525a87024a44c79dbd', 'has_outrights': True, 'sport_key': 'basketball_ncaab_championship_winner', 'sport_title': 'NCAAB Championship Winner', 'commence_time': '2025-04-10T18:00:00Z', 'home_team': None, 'away_team': None, 'bookmakers': [{'key': 'betrivers', 'title': 'BetRivers', 'last_update': '2024-06-14T05:14:57Z', 'markets': [{'key': 'outrights', 'last_update': '2024-06-14T05:14:57Z', 'outcomes': [{'name': 'UConn Huskies', 'price': 8.0}, {'name': 'Kansas Jayhawks', 'price': 12.0}, {'name': 'Duke Blue Devils', 'price': 13.0}, {'name': 'Alabama Crimson Tide', 'price': 16.0}, {'name': 'North Carolina Tar Heels', 'price': 17.0}, {'name': 'Houston Cougars', 'price': 17.0}, {'name': 'Gonzaga Bulldogs', 'price': 21.0}, {'name': 'Arizona Wildcats', 'price': 21.0}, {'name': 'Baylor Bears', 'price': 23.0}, {'name': 'Arkansas Razorbacks', 'price': 31.0}, {'name': 'Auburn Tigers', 'price': 31.0}, {'name': 'Iowa State Cyclones', 'price': 31.0}, {'name': '

In [7]:
odds_meta = [
    ['id'],
    ['has_outrights'],
    ['sport_key'],
    ['sport_title'],
    ['commence_time'],
    ['home_team'],
    ['away_team'],
    ['bookmakers', 'key'],
    ['bookmakers', 'title'],
    ['bookmakers', 'last_update'],
    ['bookmakers', 'markets', 'key'],
    ['bookmakers', 'markets', 'last_update']
]

df = pd.json_normalize(odds_json, record_path=['bookmakers', 'markets', 'outcomes'], 
            meta=odds_meta, sep='_'
            )

In [2]:
import duckdb

conn = duckdb.connect('X:\\nba_data\\odds_data\\odds.db')

In [15]:
import duckdb
import pandas as pd

%reload_ext sql
conn = duckdb.connect('X:\\nba_data\\odds_data\\odds.db')
%sql conn --alias duckdb

%config SqlMagic.displaylimit = 25

Tip: You may define configurations in C:\Users\cwood\Documents\bball_proj\wood-ball\pyproject.toml or C:\Users\cwood\.jupysql\config.

Did not find user configurations in C:\Users\cwood\Documents\bball_proj\wood-ball\pyproject.toml.

In [28]:
%%sql
SELECT distinct bookmakers_title
FROM ncaa_champ_odds limit 5

Running query in 'duckdb'

bookmakers_title
BetRivers
BetOnline.ag
DraftKings
FanDuel
BetMGM


In [ ]:
RANK() OVER (ORDER BY price) AS rank

In [44]:
%%sql
with first as (
    select
    name, 
    avg(price) as avg_odds
    from ncaa_champ_odds
    where bookmakers_title in ('DraftKings', 'BetMGM', 'FanDuel')
    group by name
    order by avg_odds asc
    limit 25
)
select 
stddev(avg_odds) as st_dev
from first

Running query in 'duckdb'

st_dev
17.2349451212742


In [30]:
%%sql
with first as (
    select
    name, 
    avg(price) as avg_odds,
    from ncaa_champ_odds
    where bookmakers_title in ('DraftKings', 'BetMGM', 'FanDuel')
    group by name
    order by avg_odds asc
)
select 
RANK() OVER (ORDER BY avg_odds) AS rank,
*
from first

Running query in 'duckdb'

rank,name,avg_odds,stddev_odds
1,Kansas Jayhawks,11.0,0.0
1,UConn Huskies,11.0,0.0
3,Alabama Crimson Tide,13.0,2.0
4,Duke Blue Devils,13.333333333333334,0.5773502691896258
5,Houston Cougars,17.666666666666668,1.154700538379251
6,North Carolina Tar Heels,18.666666666666668,2.081665999466132
7,Baylor Bears,21.0,0.0
7,Gonzaga Bulldogs,21.0,0.0
9,Arizona Wildcats,26.0,0.0
10,Arkansas Razorbacks,26.666666666666668,4.04145188432738


In [ ]:
COPY
    (SELECT * FROM tbl)
    TO 'result-snappy.parquet'
    (FORMAT 'parquet');

In [10]:
conn.sql("create table ncaa_champ_odds as select * from df")

In [31]:
df = conn.sql("""
with first as (
    select
    name, 
    avg(price) as avg_odds,
    stddev(price) as stddev_odds
    from ncaa_champ_odds
    where bookmakers_title in ('DraftKings', 'BetMGM', 'FanDuel')
    group by name
    order by avg_odds asc
)
select 
RANK() OVER (ORDER BY avg_odds) AS rank,
*
from first
         """).df()

In [34]:
df = df.head(25)

In [35]:
from sklearn.cluster import KMeans

# Reshape the data to a 2D array for KMeans
X = df['avg_odds'].values.reshape(-1, 1)

# Choose the number of clusters (k)
k = 4

# Apply K-Means clustering
kmeans = KMeans(n_clusters=k, random_state=0).fit(X)

# Assign the cluster labels to your original DataFrame
df['cluster'] = kmeans.labels_

c:\Users\cwood\Documents\bball_proj\wood-ball\.pixi\envs\default\Lib\site-packages\sklearn\cluster\_kmeans.py:1426: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\cwood\AppData\Local\Temp\ipykernel_26740\1558134970.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['cluster'] = kmeans.labels_


In [36]:
df

,rank,name,avg_odds,stddev_odds,cluster
0,1,Kansas Jayhawks,11.000000,0.000000,3
1,1,UConn Huskies,11.000000,0.000000,3
2,3,Alabama Crimson Tide,13.000000,2.000000,3
3,4,Duke Blue Devils,13.333333,0.577350,3
4,5,Houston Cougars,17.666667,1.154701,3
5,6,North Carolina Tar Heels,18.666667,2.081666,3
6,7,Baylor Bears,21.000000,0.000000,1
7,7,Gonzaga Bulldogs,21.000000,0.000000,1
8,9,Arizona Wildcats,26.000000,0.000000,1
9,10,Arkansas Razorbacks,26.666667,4.041452,1
